# Machine Learning sur du texte avec TF-IDF et LSA

In [ ]:
!pip install --upgrade 'imbalanced-learn'
import imblearn
from IPython.display import display
import matplotlib.pyplot as plt
import numpy
import pandas
import pathlib
import seaborn
import sklearn.decomposition
import sklearn.ensemble
import sklearn.feature_extraction
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import sklearn.pipeline
import sklearn.svm

## Récupération des données
Exécutez la cellule suivante pour récupérer les deux corpus de ce TP.

Le premier est une collection de trames de scénarios de différents films ainsi que des méta-données telles que l'année de sortie, le titre ou le genre.

Le second, 20 Newsgroups, rassemble les messages de 20 forums de discussion : il servira à l'analyse sémantique latente.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh wikipedia-movie-plots
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
chemin_films = "wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Chargement dans un dataframe pandas

Chargez dans un dataframe panda les données contenues dans le fichier .csv
Vérifiez que les colonnes sont cohérentes avec le .csv
Combien y a-t-il de films  dans le base ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = pandas.read_csv(chemin_films)
print(f"Colonnes : {', '.join(df.columns)}")
print(f"Forme de la df : {df.shape}")

## Sélection de données pour la suite

Pour des raisons de représentativité, on ne va conserver que les 5 genres les plus représentés. On ne conservera évidemment pas le genre le plus représenté : `unknown`

Quels sont les genres conservés ?

Construisez un nouveau dataframe contenant uniquement ces genre.

Combien nous reste-t-il de films ?

Fonctions utiles :

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)

In [ ]:
# Votre code ici

### Solution

In [ ]:
comptes = df["Genre"].value_counts().drop("unknown")
genres = comptes.index.values[:5]
print(f"Genres conservés : {', '.join(genres)}")
print(f"Nombre de films conservés : {comptes[genres].sum()}")

In [ ]:
df_propre = df[df["Genre"].isin(genres)]
seaborn.countplot(df_propre, x="Genre")

## Transformation TF-IDF

Utilisez [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) pour convertir les textes en vecteurs.

Faites en sorte de supprimer les mots vides et de ne conserver que les 3000 mots les plus fréquents.

In [ ]:
x = df_propre["Plot"]

# Votre code ici

### Solution

In [ ]:
vectoriseur = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000
)
X = vectoriseur.fit_transform(x)
noms_variables = vectoriseur.get_feature_names_out()
print(f"10 premières colonnes : {', '.join(noms_variables[:10])}")

In [ ]:
indices_0 = X[0].indices
mots_0 = [vectoriseur.get_feature_names_out()[i] for i in X[0].indices]
print(f"Colonnes > 0 dans la première ligne de X : {indices_0}")
print(f"Mots correspondants : {', '.join(mots_0)}")
print(f"Première valeur de x : {x.iloc[0]}")

## Un corpus pour la LSA

L'analyse sémantique latente (LSA) est une SVD tronquée de la matrice documents × mots : chaque axe regroupe des mots qui apparaissent ensemble. Pour qu'elle montre quelque chose, il faut un corpus aux thèmes bien marqués. Ce n'est pas le cas des résumés de films : leurs axes se remplissent de prénoms (`joe`, `jack`, `mary`). On passe donc à 20 Newsgroups, dont les messages viennent de 20 forums thématiques : sport, informatique, religion, politique…

Exécutez la cellule suivante. Elle charge les 11 314 messages de la partie entraînement du corpus, et leur retire leur en-tête (expéditeur, sujet…), qui s'arrête à la première ligne vide.

In [ ]:
# Chaque dossier correspond à un forum, puis un message par fichier
messages = [
  p.read_text(encoding="latin-1").partition("\n\n")[2]
  for p in sorted(pathlib.Path("20news-bydate-train").glob("*/*"))
]

print(f"{len(messages)} messages récupérés")
print(messages[0][:300])

## Transformation TF-IDF des messages

Comme pour les films, convertissez `messages` en vecteurs avec `TfidfVectorizer` : supprimez les mots vides et gardez les 3000 mots les plus fréquents. Retirez aussi les mots présents dans plus de 30 % des messages, ou dans moins de 5 messages.

Affichez la taille de la matrice et le nombre moyen de valeurs non nulles par ligne : combien des 3000 mots un message emploie-t-il ?

**Documentation utile**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`stop_words` et `max_features` s'utilisent comme pour les films. `max_df=0.3` retire les mots présents dans plus de 30 % des documents (un nombre à virgule est une part des documents) et `min_df=5` ceux présents dans moins de 5 documents (un entier est un nombre de documents).

`X_messages.shape` donne la taille de la matrice et `X_messages.nnz` son nombre de valeurs non nulles. Vous devez obtenir `(11314, 3000)` et environ 55 mots par message.

```python
vectoriseur_messages = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words=..., max_features=..., max_df=..., min_df=...
)
X_messages = vectoriseur_messages.fit_transform(...)
noms_variables_messages = vectoriseur_messages.get_feature_names_out()
print(f"Taille de la matrice : {X_messages.shape}")
print(f"Mots par message : {...:.0f}")
```

### Solution

In [ ]:
vectoriseur_messages = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000, max_df=0.3, min_df=5
)
X_messages = vectoriseur_messages.fit_transform(messages)
noms_variables_messages = vectoriseur_messages.get_feature_names_out()
print(f"Taille de la matrice : {X_messages.shape}")
print(f"Mots par message : {X_messages.nnz / X_messages.shape[0]:.0f}")

## Analyse sémantique latente

Utilisez `TruncatedSVD` pour calculer une LSA à 6 axes sur `X_messages`.

Affichez la valeur singulière de chaque axe, la part de variance qu'il conserve, et la part que conservent les 6 axes ensemble. Quelle part de la variance gardent-ils ? L'axe qui a la plus grande valeur singulière garde-t-il la plus grande variance ?

**Documentation utile**

- [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`n_components=6` fixe le nombre d'axes, `n_iter=100` affine le calcul et `random_state=42` redonne les mêmes axes d'une exécution à l'autre. Après `svd.fit(X_messages)`, les attributs `singular_values_` et `explained_variance_ratio_` ont une valeur par axe.

Vous devez obtenir une valeur singulière de 16 pour l'axe 1, et environ 2,8 % de variance pour les 6 axes.

```python
svd = sklearn.decomposition.TruncatedSVD(...)
svd.fit(...)

print(f"Valeurs singulières : {svd.singular_values_.round(2)}")
print(f"Part de variance par axe : {...}")
print(f"Part de variance des 6 axes : {...:.1%}")
```

### Solution

In [ ]:
svd = sklearn.decomposition.TruncatedSVD(n_components=6, n_iter=100, random_state=42)
svd.fit(X_messages)

print(f"Valeurs singulières : {svd.singular_values_.round(2)}")
print(f"Part de variance par axe : {svd.explained_variance_ratio_.round(4)}")
print(f"Part de variance des 6 axes : {svd.explained_variance_ratio_.sum():.1%}")

## Mots importants

Les 6 axes gardent moins de 3 % de la variance : c'est normal, et ce n'est pas un échec. Un message n'emploie qu'environ 55 des 3000 mots, et l'essentiel de la variance vient de ces choix propres à chaque message : il faudrait environ 500 axes pour en garder la moitié. La LSA ne retient que ce que beaucoup de messages partagent : un axe se juge à ses mots, pas à sa part de variance.

L'axe 1 n'est pas un thème. `TruncatedSVD` ne centre pas la matrice, dont aucune valeur n'est négative : cet axe a tous ses coefficients positifs, et il donne le vocabulaire moyen du corpus. Il a ainsi la plus grande valeur singulière sans garder la plus grande variance. Les thèmes se lisent à partir de l'axe 2.

Le signe d'un coefficient porte le sens : un axe oppose le pôle de ses coefficients positifs à celui de ses coefficients négatifs. Pour chaque axe, affichez les 8 mots les plus positifs et les 8 mots les plus négatifs. Puis, pour les axes 2 à 6, nommez chaque pôle : de quel thème parle-t-il ?

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`svd.components_` a une ligne par axe et une colonne par mot. `numpy.argsort(svd.components_, axis=1)` trie chaque ligne : pour un axe, `indices` va du coefficient le plus négatif au plus positif. `indices[:8]` donne donc les 8 plus négatifs, `indices[::-1][:8]` les 8 plus positifs, et `noms_variables_messages[indices[:8]]` les mots correspondants.

Vous devez trouver `god` et `jesus` à un bout de l'axe 2, `windows` et `card` à l'autre. Seule l'opposition compte : quel pôle est positif ne dépend que du calcul.

```python
nb_mots = 8

for numero, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positifs = ...
  negatifs = ...
  print(f"Axe {numero}")
  print(f"  + {', '.join(positifs)}")
  print(f"  - {', '.join(negatifs)}")
```

### Solution

In [ ]:
nb_mots = 8

# argsort trie les coefficients de chaque axe, du plus négatif au plus positif
for numero, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positifs = noms_variables_messages[indices[::-1][:nb_mots]]
  negatifs = noms_variables_messages[indices[:nb_mots]]
  print(f"Axe {numero}")
  print(f"  + {', '.join(positifs)}")
  print(f"  - {', '.join(negatifs)}")

# Axe 1 : le vocabulaire moyen. Ses coefficients sont tous positifs : les mots
# affichés comme négatifs sont juste les plus proches de 0.
# Axe 2 : religion contre informatique
# Axe 3 : religion contre sport
# Axe 4 : cryptographie contre religion et sport
# Axe 5 : matériel contre logiciel
# Axe 6 : cryptographie contre Proche-Orient
# Les pôles ne sont pas purs (`windows` côté religion sur l'axe 3) : les axes
# doivent rester orthogonaux, si bien qu'un thème se répartit sur plusieurs axes.

## Approche supervisée

Retour aux films : on va essayer de prédire le genre du film à partir du scénario.

Pour cela, commencez par extraire le genre de chaque film dans une série pandas `y`. Les classifieurs de scikit-learn prennent ces chaînes de caractères telles quelles.

`drama` compte 6 fois plus de films que `thriller`. Quelle justesse obtient un modèle qui répond toujours `drama` ? `value_counts(normalize=True)` donne la part de chaque genre. C'est le score à battre.

In [ ]:
# Votre code ici

### Solution

In [ ]:
y = df_propre["Genre"]

parts = y.value_counts(normalize=True)
display(parts)

print(f"Justesse en répondant toujours drama : {parts['drama']:.1%}")

## Apprentissage

Utilisez un algorithme de classification de votre choix pour apprendre un modèle qui prend en entrée les vecteurs tf-idf des scénarios et prédit la bonne classe.

Évaluez-le par validation croisée ([`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)). Le vocabulaire et les poids idf font partie du modèle : pour qu'ils ne soient appris que sur les données d'entraînement de chaque pli, mettez le `TfidfVectorizer` et le classifieur dans un même pipeline ([`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)), appliqué aux textes `x`. Le fichier est trié par origine des films : pour que les plis ne suivent pas ces blocs, mélangez-les avec [`sklearn.model_selection.StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).

Affichez la justesse, les précisions micro et macro, ainsi que la matrice de confusion correspondante.

In [ ]:
# Votre code ici

### Solution

In [ ]:
def evalue_pipeline(modele):
  # Le vectoriseur fait partie du pipeline : à chaque pli de la validation
  # croisée, il n'apprend son vocabulaire et ses poids idf que sur les données
  # d'entraînement du pli
  modele = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(
      stop_words="english", max_features=3000
    ),
    modele,
  )
  # Le fichier est trié par origine : on mélange pour que les plis ne suivent pas
  # ces blocs
  plis = sklearn.model_selection.StratifiedKFold(5, shuffle=True, random_state=0)
  predictions = sklearn.model_selection.cross_val_predict(modele, x, y, cv=plis)

  # La justesse est le nombre de bonnes prédictions sur le nombre de prédictions
  justesse = sklearn.metrics.accuracy_score(y, predictions)

  # La micro-précision cumule les vrais et faux positifs de toutes les classes
  # avant de diviser : quand chaque exemple a une seule classe, elle est égale
  # à la justesse
  precision_micro = sklearn.metrics.precision_score(y, predictions, average="micro")

  # La macro-précision est la moyenne non pondérée des précisions de chaque
  # classe : les petites classes y comptent autant que les grandes
  precision_macro = sklearn.metrics.precision_score(y, predictions, average="macro")

  print(
    f"Justesse : {justesse:.2f}, "
    f"micro précision : {precision_micro:.2f}, "
    f"macro précision : {precision_macro:.2f}"
  )

  matrice_confusion = sklearn.metrics.confusion_matrix(y, predictions, normalize="true")
  seaborn.heatmap(
    matrice_confusion,
    vmin=0,
    vmax=1,
    xticklabels=sorted(genres),
    yticklabels=sorted(genres),
    cmap=seaborn.color_palette("Blues", as_cmap=True),
  )
  plt.plot()

In [ ]:
# Avec une régression logistique
evalue_pipeline(sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200))

In [ ]:
# Avec plusieurs régressions logistiques bootstrappées avec sous-échantillonage
evalue_pipeline(
  imblearn.ensemble.BalancedBaggingClassifier(
    sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200)
  )
)

In [ ]:
# Avec sur-échantillonage aléatoire
evalue_pipeline(
  imblearn.pipeline.make_pipeline(
    imblearn.over_sampling.RandomOverSampler(),
    sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200),
  )
)

In [ ]:
# Avec un SVM rapide
# evalue_pipeline(sklearn.svm.LinearSVC(class_weight="balanced"))

In [ ]:
# Avec une random forest
# evalue_pipeline(sklearn.ensemble.RandomForestClassifier(n_estimators=100,
#                                                         class_weight="balanced"))

In [ ]:
# Avec un SVM long à entraîner mais au kernel plus puissant
# evalue_pipeline(sklearn.svm.SVC(class_weight="balanced"))